# LeetCode 3435 — Frequencies of Shortest Supersequences bằng Python

- Link: https://leetcode.com/problems/frequencies-of-shortest-supersequences/
- Độ khó: **Hard**
- Chủ đề: **Đồ thị có hướng** từ các cặp chữ, **mỗi chữ xuất hiện 1 hoặc 2 lần**, liệt kê **bitmask** tập chữ "nhân đôi", kiểm tra **đồ thị con không chu trình** (sắp xếp tô-pô bằng bitmask)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def supersequences(self, words: List[str]) -> List[List[int]]:
```

## 1. Mục tiêu học

1. Đưa bài "xâu cha chung ngắn nhất (SCS) của các từ độ dài 2" về **đồ thị**: từ `xy` là cạnh `x → y` (đòi `x` đứng trước `y` ở đâu đó).
2. Chứng minh trong SCS mỗi chữ xuất hiện **1 hoặc 2 lần**, và một chữ xuất hiện 2 lần có thể "thỏa" mọi cạnh liên quan tới nó.
3. Điều kiện khả thi: tập chữ xuất hiện **1 lần** phải tạo đồ thị con **không chu trình** (và không có từ dạng `xx`).
4. Duyệt tập con các chữ nhân đôi theo **số phần tử tăng dần**, dừng ở mức nhỏ nhất; xuất mọi vectơ tần số ở mức đó.
5. Kiểm tra không chu trình nhanh bằng **Kahn trên bitmask**, và cách DP "tập không chu trình" cho mọi mặt nạ.

## 2. Đề bài & ví dụ

Cho các xâu `words` (mỗi xâu độ dài 2, tất cả khác nhau, dùng ≤ 16 chữ cái). Tìm mọi **xâu cha chung ngắn nhất** (chứa mọi từ như dãy con), **bỏ các xâu là hoán vị của nhau**, trả về vectơ tần số 26 phần tử của chúng (thứ tự bất kỳ).

**Ví dụ 1:** `["ab","ba"]` → tần số của `"aba"` và `"bab"`.

**Ví dụ 2:** `["aa","ac"]` → chỉ tần số của `"aac"` (`"aca"` là hoán vị).

**Ví dụ 3:** `["aa","bb","cc"]` → tần số của `"aabbcc"`.

**Ràng buộc:** `1 ≤ |words| ≤ 256`.

In [1]:
from typing import List

def vec(counts):
    f = [0] * 26
    for ch, c in counts.items():
        f[ord(ch) - 97] = c
    return f

examples = [
    (["ab", "ba"], sorted([vec({'a': 1, 'b': 2}), vec({'a': 2, 'b': 1})])),
    (["aa", "ac"], [vec({'a': 2, 'c': 1})]),
    (["aa", "bb", "cc"], [vec({'a': 2, 'b': 2, 'c': 2})]),
]
for w, exp in examples:
    print(w, '->', [{chr(97 + i): v for i, v in enumerate(f) if v} for f in exp])

['ab', 'ba'] -> [{'a': 1, 'b': 2}, {'a': 2, 'b': 1}]
['aa', 'ac'] -> [{'a': 2, 'c': 1}]
['aa', 'bb', 'cc'] -> [{'a': 2, 'b': 2, 'c': 2}]


## 3. Kiến thức nền

### 3.1. Dãy con

In [2]:
def is_subseq(w, t):
    it = iter(t)
    return all(c in it for c in w)

print(is_subseq("ab", "aba"), is_subseq("ba", "aba"), is_subseq("ba", "ab"))

True True False


### 3.2. Mỗi chữ xuất hiện tối đa 2 lần

Gọi `U` là tập chữ. Xâu `x_1 x_2 … x_m  x_1 x_2 … x_m` (mọi chữ, rồi lặp lại) chứa mọi cặp `xy` như dãy con (lấy `x` ở nửa đầu, `y` ở nửa sau). Trong một SCS, nếu chữ `x` xuất hiện ≥ 3 lần, giữ lần **đầu** và **cuối**, bỏ các lần giữa: mọi cặp `x?` dùng lần đầu, mọi cặp `?x` dùng lần cuối, cặp `xx` dùng cả hai — vẫn là xâu cha nhưng ngắn hơn, mâu thuẫn. ⇒ mỗi chữ xuất hiện 1 hoặc 2 lần.

### 3.3. Đồ thị và sắp xếp tô-pô bằng bitmask

Cạnh `x → y` (từ `xy`, `x ≠ y`). Tập `S` **không chu trình** ⇔ lặp lại việc xóa các đỉnh không có cạnh **vào** từ phần còn lại của `S` cho tới khi rỗng. Lưu `inc[v]` = mặt nạ các đỉnh có cạnh tới `v`.

In [3]:
def acyclic(S, inc):
    rem = S
    while rem:
        moved = False
        r = rem
        while r:
            v = (r & -r).bit_length() - 1
            r &= r - 1
            if inc[v] & rem == 0:
                rem &= ~(1 << v)
                moved = True
        if not moved:
            return False
    return True

# đồ thị 0 -> 1 -> 2 -> 0 (chu trình), 3 độc lập
inc = [1 << 2, 1 << 0, 1 << 1, 0]
print(acyclic(0b1111, inc), acyclic(0b1011, inc), acyclic(0b0111 & ~0b100, inc))

False True True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi `D` là tập chữ xuất hiện **2 lần**, `U \ D` xuất hiện 1 lần. Với `D` cho trước, xâu

`(các chữ của D)  +  (các chữ của U \ D theo thứ tự tô-pô)  +  (các chữ của D)`

thỏa mọi từ `xy` nếu:

- `x ∈ D`: dùng bản sao **đầu** của `x`, `y` đứng sau ở đâu đó (giữa hoặc bản sao cuối);
- `y ∈ D`: dùng bản sao **cuối** của `y`;
- cả hai thuộc `U \ D`: cần `x` trước `y` trong thứ tự tô-pô ⇒ đồ thị con trên `U \ D` phải **không chu trình**;
- từ `xx`: bắt buộc `x ∈ D`.

Độ dài `|U| + |D|` ⇒ SCS ứng với `D` **nhỏ nhất** thỏa điều kiện. Ví dụ 1: `a → b → a` là chu trình ⇒ cần `|D| ≥ 1`; `D = {a}` hoặc `D = {b}` đều được ⇒ hai vectơ tần số.

## 5. Cách 1 — Brute Force: thử mọi xâu theo độ dài tăng dần

In [4]:
import itertools

def scs_brute(words):
    letters = sorted(set(''.join(words)))
    for L in range(len(letters), 2 * len(letters) + 1):
        res = set()
        for tup in itertools.product(letters, repeat=L):
            t = ''.join(tup)
            if all(is_subseq(w, t) for w in words):
                f = [0] * 26
                for c in t:
                    f[ord(c) - 97] += 1
                res.add(tuple(f))
        if res:
            return sorted(map(list, res))

for w, exp in examples:
    print(scs_brute(w) == exp)

True
True
True


**Độ phức tạp:** `O(|U|^{2|U|})` — chỉ dùng cho ≤ 3–4 chữ cái.

## 6. Ý tưởng thuật toán / bước đột phá

1. Đánh số chữ `0..m−1` (`m ≤ 16`), dựng `inc[v]` và mặt nạ `self` các chữ có từ `xx`.
2. Duyệt mọi `D ⊆ U` theo **số phần tử tăng dần**; bỏ `D` không chứa `self`.
3. `D` hợp lệ ⇔ `U \ D` không chu trình.
4. Gặp mức `|D|` đầu tiên có lời giải: thu **mọi** `D` hợp lệ ở mức đó, xuất vectơ tần số (1 cho chữ đơn, 2 cho chữ trong `D`), rồi dừng.

Hai `D` khác nhau cho vectơ khác nhau, và hai SCS là hoán vị của nhau ⇔ cùng vectơ ⇔ cùng `D`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi `D`, kiểm tra chu trình bằng DFS

In [5]:
def build(words):
    letters = sorted(set(''.join(words)))
    idx = {c: i for i, c in enumerate(letters)}
    m = len(letters)
    out = [set() for _ in range(m)]
    selfm = 0
    for w in words:
        a, b = idx[w[0]], idx[w[1]]
        if a == b:
            selfm |= 1 << a
        else:
            out[a].add(b)
    return letters, out, selfm

def has_cycle_dfs(nodes, out):
    color = {v: 0 for v in nodes}
    def dfs(u):
        color[u] = 1
        for v in out[u]:
            if v in color:
                if color[v] == 1 or (color[v] == 0 and dfs(v)):
                    return True
        color[u] = 2
        return False
    return any(color[v] == 0 and dfs(v) for v in nodes)

def freq_of(letters, D):
    f = [0] * 26
    for i, c in enumerate(letters):
        f[ord(c) - 97] = 2 if D >> i & 1 else 1
    return f

def scs_all_masks(words):
    letters, out, selfm = build(words)
    m = len(letters)
    best, res = None, []
    for D in range(1 << m):
        if selfm & ~D:
            continue
        singles = [v for v in range(m) if not D >> v & 1]
        if has_cycle_dfs(singles, out):
            continue
        pc = bin(D).count('1')
        if best is None or pc < best:
            best, res = pc, [freq_of(letters, D)]
        elif pc == best:
            res.append(freq_of(letters, D))
    return sorted(res)

for w, exp in examples:
    print(scs_all_masks(w) == exp)

True
True
True


### 7.2. Cách 3 — Duyệt theo số phần tử tăng dần + Kahn bitmask, dừng sớm (lời giải chính)

In [6]:
def scs_by_popcount(words):
    letters, out, selfm = build(words)
    m = len(letters)
    inc = [0] * m
    for a in range(m):
        for b in out[a]:
            inc[b] |= 1 << a
    full = (1 << m) - 1
    order = sorted(range(1 << m), key=lambda x: bin(x).count('1'))
    res, best = [], None
    for D in order:
        pc = bin(D).count('1')
        if best is not None and pc > best:
            break
        if selfm & ~D:
            continue
        if acyclic(full & ~D, inc):
            best = pc
            res.append(freq_of(letters, D))
    return sorted(res)

for w, exp in examples:
    print(scs_by_popcount(w) == exp)

True
True
True


### 7.3. Cách 4 — DP "tập không chu trình" cho mọi mặt nạ

`acyc[S] = True` ⇔ tồn tại `v ∈ S` không có cạnh vào từ `S \ {v}` **và** `acyc[S \ {v}]` (đỉnh đầu của một thứ tự tô-pô). Tính tăng dần theo mặt nạ trong `O(2^m · m)`, rồi chọn các `D` nhỏ nhất với `acyc[full ^ D]`.

In [7]:
def scs_dp(words):
    letters, out, selfm = build(words)
    m = len(letters)
    inc = [0] * m
    for a in range(m):
        for b in out[a]:
            inc[b] |= 1 << a
    acyc = bytearray(1 << m)
    acyc[0] = 1
    for S in range(1, 1 << m):
        r = S
        while r:
            v = (r & -r).bit_length() - 1
            r &= r - 1
            if inc[v] & S == 0 and acyc[S ^ (1 << v)]:
                acyc[S] = 1
                break
    full = (1 << m) - 1
    best = min(bin(D).count('1') for D in range(1 << m) if not (selfm & ~D) and acyc[full ^ D])
    return sorted(freq_of(letters, D) for D in range(1 << m)
                  if bin(D).count('1') == best and not (selfm & ~D) and acyc[full ^ D])

for w, exp in examples:
    print(scs_dp(w) == exp)

True
True
True


## 8. TRACE từng bước

Trace cách 3 trên `["ab", "bc", "ca", "dd"]`: chu trình `a → b → c → a` và từ `dd` bắt buộc nhân đôi `d`.

In [8]:
def trace(words):
    letters, out, selfm = build(words)
    m = len(letters)
    inc = [0] * m
    for a in range(m):
        for b in out[a]:
            inc[b] |= 1 << a
    full = (1 << m) - 1
    print('chữ:', letters, '| bắt buộc nhân đôi:', [letters[i] for i in range(m) if selfm >> i & 1])
    best = None
    for D in sorted(range(1 << m), key=lambda x: bin(x).count('1')):
        pc = bin(D).count('1')
        if best is not None and pc > best:
            break
        names = ''.join(letters[i] for i in range(m) if D >> i & 1) or '∅'
        if selfm & ~D:
            print(f"D = {names:5s}: thiếu chữ bắt buộc"); continue
        ok = acyclic(full & ~D, inc)
        print(f"D = {names:5s}: phần còn lại {'không chu trình -> HỢP LỆ' if ok else 'có chu trình'}")
        if ok:
            best = pc
    print('kết quả:', [{letters[i]: f[ord(letters[i]) - 97] for i in range(m)} for f in scs_by_popcount(words)])

trace(["ab", "bc", "ca", "dd"])

chữ: ['a', 'b', 'c', 'd'] | bắt buộc nhân đôi: ['d']
D = ∅    : thiếu chữ bắt buộc
D = a    : thiếu chữ bắt buộc
D = b    : thiếu chữ bắt buộc
D = c    : thiếu chữ bắt buộc
D = d    : phần còn lại có chu trình
D = ab   : thiếu chữ bắt buộc
D = ac   : thiếu chữ bắt buộc
D = bc   : thiếu chữ bắt buộc
D = ad   : phần còn lại không chu trình -> HỢP LỆ
D = bd   : phần còn lại không chu trình -> HỢP LỆ
D = cd   : phần còn lại không chu trình -> HỢP LỆ
kết quả: [{'a': 1, 'b': 1, 'c': 2, 'd': 2}, {'a': 1, 'b': 2, 'c': 1, 'd': 2}, {'a': 2, 'b': 1, 'c': 1, 'd': 2}]


## 9. Code nộp LeetCode

In [9]:
from itertools import combinations

class Solution:
    def supersequences(self, words: List[str]) -> List[List[int]]:
        letters = sorted(set(''.join(words)))
        idx = {c: i for i, c in enumerate(letters)}
        m = len(letters)
        inc = [0] * m
        selfm = 0
        for w in words:
            a, b = idx[w[0]], idx[w[1]]
            if a == b:
                selfm |= 1 << a
            else:
                inc[b] |= 1 << a
        full = (1 << m) - 1

        def acyclic(S):
            rem = S
            while rem:
                moved = False
                r = rem
                while r:
                    v = (r & -r).bit_length() - 1
                    r &= r - 1
                    if inc[v] & rem == 0:
                        rem &= ~(1 << v)
                        moved = True
                if not moved:
                    return False
            return True

        res = []
        for size in range(m + 1):
            for comb in combinations(range(m), size):
                D = 0
                for v in comb:
                    D |= 1 << v
                if selfm & ~D:
                    continue
                if acyclic(full & ~D):
                    f = [0] * 26
                    for i, c in enumerate(letters):
                        f[ord(c) - 97] = 2 if D >> i & 1 else 1
                    res.append(f)
            if res:
                return res
        return res

for w, exp in examples:
    got = Solution().supersequences(w)
    print(sorted(got) == exp)

True
True
True


## 10. Vì sao thuật toán đúng

1. **Mỗi chữ 1–2 lần (3.2)** và mọi chữ trong `U` phải xuất hiện ⇒ SCS xác định bởi tập `D` (các chữ xuất hiện 2 lần) và độ dài `|U| + |D|`.
2. **Điều kiện đủ (mục 4):** nếu `U \ D` không chu trình và chứa không từ `xx` nào, xây dựng `D + topo(U\D) + D` là xâu cha.
3. **Điều kiện cần:** trong một xâu cha bất kỳ, các chữ xuất hiện đúng một lần có vị trí duy nhất; mọi cạnh `x → y` giữa chúng buộc `pos(x) < pos(y)` ⇒ không thể có chu trình; từ `xx` cần ≥ 2 lần.
4. **Không lặp hoán vị:** vectơ tần số ↔ tập `D` là song ánh. Mỗi `D` nhỏ nhất hợp lệ cho đúng một vectơ, và mọi SCS đều có vectơ thuộc tập này.

## 11. Phân tích Time / Space Complexity

`m ≤ 16`, `E ≤ 256` cạnh.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi xâu | `O(m^{2m})` | `O(m)` |
| 2. Mọi `D` + DFS | `O(2^m · (m + E))` | `O(m + E)` |
| 3. Theo cỡ + Kahn bitmask, dừng sớm | `O(Σ_{|D| ≤ k*} C(m, |D|) · m²)` | `O(m)` |
| 4. DP tập không chu trình | `O(2^m · m)` | `O(2^m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'all_masks': scs_all_masks, 'by_popcount': scs_by_popcount, 'dp': scs_dp,
         'submit': lambda w: sorted(Solution().supersequences(w))}
edge = [
    (["ab"], [vec({'a': 1, 'b': 1})]),
    (["aa"], [vec({'a': 2})]),
    (["ab", "bc", "ca"], sorted([vec({'a': 2, 'b': 1, 'c': 1}), vec({'a': 1, 'b': 2, 'c': 1}), vec({'a': 1, 'b': 1, 'c': 2})])),
    (["ab", "ba", "aa"], [vec({'a': 2, 'b': 1})]),
]
for w, exp in examples + edge:
    assert scs_brute(w) == exp, (w, scs_brute(w))
    for name, f in impls.items():
        assert f(w) == exp, (name, w)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3435)
cnt = 0
for _ in range(300):
    al = 'abc'[:rng.randint(1, 3)]
    pairs = sorted({a + b for a in al for b in al})
    w = rng.sample(pairs, rng.randint(1, len(pairs)))
    exp = scs_brute(w)
    for name, f in impls.items():
        assert f(w) == exp, (name, w)
    cnt += 1
for _ in range(25):                                    # 4 chữ cái (vét cạn chậm hơn)
    pairs = sorted({a + b for a in 'abcd' for b in 'abcd'})
    w = rng.sample(pairs, rng.randint(1, 6))
    assert scs_by_popcount(w) == scs_brute(w) == scs_dp(w)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

for _ in range(200):                                   # 16 chữ cái: các cách nhanh khớp nhau
    al = 'abcdefghijklmnop'[:rng.randint(5, 16)]
    pairs = [a + b for a in al for b in al]
    w = rng.sample(pairs, rng.randint(1, min(60, len(pairs))))
    assert scs_by_popcount(w) == scs_dp(w) == sorted(Solution().supersequences(w))
print('Cách 3, 4 và bài nộp khớp nhau trên 200 bộ tới 16 chữ cái.')

Khớp vét cạn trên 325 bộ ngẫu nhiên.


Cách 3, 4 và bài nộp khớp nhau trên 200 bộ tới 16 chữ cái.


### 12.3. Ca lớn: 16 chữ cái, tới 256 từ

In [12]:
import time
al = 'abcdefghijklmnop'
cases = {
    'đầy đủ 256 từ': [a + b for a in al for b in al],
    'đầy đủ, bỏ xx (240 từ)': [a + b for a in al for b in al if a != b],
    'ngẫu nhiên 120 từ': rng.sample([a + b for a in al for b in al if a != b], 120),
    'chu trình dài + thưa': [al[i] + al[(i + 1) % 16] for i in range(16)] + [al[i] + al[(i + 5) % 16] for i in range(16)],
}
for name, w in cases.items():
    t = time.time(); r = Solution().supersequences(w); t1 = time.time() - t
    t = time.time(); r4 = scs_dp(w); t2 = time.time() - t
    assert sorted(r) == r4
    print(f"{name:24s}: {len(r):6d} vectơ, độ dài SCS {sum(r[0])} | nộp bài {t1:.2f}s | DP {t2:.2f}s")

đầy đủ 256 từ           :      1 vectơ, độ dài SCS 32 | nộp bài 0.02s | DP 0.05s
đầy đủ, bỏ xx (240 từ)  :     16 vectơ, độ dài SCS 31 | nộp bài 0.07s | DP 0.05s


ngẫu nhiên 120 từ       :     15 vectơ, độ dài SCS 26 | nộp bài 0.07s | DP 0.06s
chu trình dài + thưa    :      4 vectơ, độ dài SCS 20 | nộp bài 0.00s | DP 0.04s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi mỗi chữ chỉ xuất hiện một lần (sắp tô-pô toàn bộ)

In [13]:
def wrong_once(words):
    letters, out, selfm = build(words)
    m = len(letters)
    if has_cycle_dfs(list(range(m)), out) or selfm:
        return []                                      # SAI: kết luận "không có SCS"
    return [freq_of(letters, 0)]

print('sai :', wrong_once(["ab", "ba"]), '| đúng:', len(Solution().supersequences(["ab", "ba"])), 'vectơ')

sai : [] | đúng: 2 vectơ


**Vì sao sai:** chu trình `a → b → a` vẫn có xâu cha (`aba`) — chỉ cần một chữ xuất hiện 2 lần.

### Lỗi 2 — Bỏ qua từ dạng `xx`

In [14]:
def wrong_ignore_self(words):
    return scs_by_popcount([w for w in words if w[0] != w[1]] or ["ab"]) if any(w[0] != w[1] for w in words) else []

w = ["aa", "ac"]
print('sai :', [{chr(97 + i): v for i, v in enumerate(f) if v} for f in wrong_ignore_self(w)],
      '| đúng:', [{chr(97 + i): v for i, v in enumerate(f) if v} for f in Solution().supersequences(w)])

sai : [{'a': 1, 'c': 1}] | đúng: [{'a': 2, 'c': 1}]


**Vì sao sai:** từ `"aa"` buộc `a` xuất hiện **hai** lần.

### Lỗi 3 — Chỉ trả về **một** SCS

In [15]:
def wrong_one(words):
    return scs_by_popcount(words)[:1]

print('sai :', len(wrong_one(["ab", "ba"])), 'vectơ | đúng:', len(Solution().supersequences(["ab", "ba"])), 'vectơ')

sai : 1 vectơ | đúng: 2 vectơ


**Vì sao sai:** đề yêu cầu **mọi** SCS (khác nhau về tần số).

### Lỗi 4 — Trả mọi `D` hợp lệ (không chỉ những `D` nhỏ nhất)

In [16]:
def wrong_all_valid(words):
    letters, out, selfm = build(words)
    m = len(letters)
    return sorted(freq_of(letters, D) for D in range(1 << m)
                  if not (selfm & ~D) and not has_cycle_dfs([v for v in range(m) if not D >> v & 1], out))

print('sai :', len(wrong_all_valid(["ab", "ba"])), 'vectơ | đúng:', len(Solution().supersequences(["ab", "ba"])), 'vectơ')

sai : 3 vectơ | đúng: 2 vectơ


**Vì sao sai:** `D = {a, b}` cũng hợp lệ (`abab`) nhưng **không ngắn nhất**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xâu cha chứa các cặp `xy` như dãy con" ⇒ đồ thị thứ tự; chu trình ⇒ cần lặp chữ.
- Ràng buộc ≤ 16 phần tử ⇒ liệt kê bitmask (65536) là khả thi.
- "Tất cả lời giải tối ưu, bỏ trùng" ⇒ tìm **biểu diễn chuẩn** (ở đây: tập `D`).
- Kiểm tra không chu trình trên tập con ⇒ Kahn với mặt nạ cạnh vào.

## 15. Biến thể của bài

1. Chỉ cần **độ dài** SCS ⇒ `|U| + (feedback vertex set nhỏ nhất, cộng các chữ có từ xx)`.
2. Từ độ dài 3 trở lên ⇒ một chữ có thể cần 3 lần; bài tổng quát (SCS của nhiều xâu) là NP-khó.
3. Đếm **số xâu** SCS (không bỏ hoán vị) ⇒ đếm thứ tự tô-pô và cách đặt các bản sao — khó hơn nhiều.
4. Bài liên quan: LC 1092 (SCS của 2 xâu — DP), LC 269 (Alien Dictionary — tô-pô).

In [17]:
# Biến thể 1: độ dài SCS = |U| + |D nhỏ nhất|
def scs_length(words):
    r = Solution().supersequences(words)
    return sum(r[0])

def scs_length_brute(words):
    letters = sorted(set(''.join(words)))
    for L in range(len(letters), 2 * len(letters) + 1):
        for tup in itertools.product(letters, repeat=L):
            if all(is_subseq(w, ''.join(tup)) for w in words):
                return L

rng = random.Random(15)
for _ in range(150):
    pairs = sorted({a + b for a in 'abc' for b in 'abc'})
    w = rng.sample(pairs, rng.randint(1, len(pairs)))
    assert scs_length(w) == scs_length_brute(w)
print('Biến thể độ dài khớp vét cạn; ví dụ 3 có độ dài', scs_length(["aa", "bb", "cc"]))

Biến thể độ dài khớp vét cạn; ví dụ 3 có độ dài 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi xâu | mũ (rất lớn) | Oracle, ≤ 4 chữ |
| 2 | Mọi `D` + DFS | `O(2^m (m + E))` | Đơn giản |
| 3 | Theo cỡ + Kahn bitmask | dừng sớm | **Nộp bài** |
| 4 | DP tập không chu trình | `O(2^m m)` | Ổn định, không phụ thuộc đáp án |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `["ab", "bc"]`.

**Bài 2 (tính tay):** `["ab", "ba", "bc", "cb"]`.

**Bài 3 (điền TODO):** viết `build_one_scs(words)` — dựng **một** xâu SCS cụ thể.

**Bài 4 (tự viết lại):** viết lại kiểm tra không chu trình bằng DFS ba màu trên mặt nạ.

**Bài 5 (mini challenge):** đếm số tập `D` nhỏ nhất mà **không** liệt kê vectơ (chỉ đếm), dùng cách 4.

In [18]:
def build_one_scs_todo(words):
    # TODO
    raise NotImplementedError

def acyclic_dfs_todo(S, out_masks):
    # TODO
    raise NotImplementedError

def count_min_D_todo(words):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `a → b → c` không chu trình. (2) `D = ∅`. (3) Một vectơ, xâu `abc`.

**Bài 2.** (1) Hai chu trình `a ↔ b`, `b ↔ c`. (2) `D = {b}` phá cả hai. (3) Thử `D = {a, c}` — cỡ 2, không nhỏ nhất.

**Bài 3.** (1) Lấy một `D` nhỏ nhất. (2) Sắp tô-pô `U \ D`. (3) Ghép `D + topo + D` và kiểm tra mọi từ là dãy con.

**Bài 4.** (1) Đổi mặt nạ cạnh ra thành danh sách. (2) Chỉ đi tới đỉnh thuộc `S`. (3) So kết quả với `acyclic`.

**Bài 5.** (1) Tính `acyc[]` như cách 4. (2) Đếm `D` có cỡ nhỏ nhất. (3) So với độ dài danh sách của bài nộp.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2
a1 = sorted(Solution().supersequences(["ab", "bc"]))
a2 = sorted(Solution().supersequences(["ab", "ba", "bc", "cb"]))
assert a1 == [vec({'a': 1, 'b': 1, 'c': 1})] and a2 == [vec({'a': 1, 'b': 2, 'c': 1})]
assert a2 == scs_brute(["ab", "ba", "bc", "cb"])
print('Bài 1:', a1[0][:3], '| Bài 2:', a2[0][:3])

# Bài 3
def build_one_scs(words):
    letters, out, selfm = build(words)
    m = len(letters)
    f = Solution().supersequences(words)[0]
    D = {c for c in letters if f[ord(c) - 97] == 2}
    singles = [i for i, c in enumerate(letters) if c not in D]
    indeg = {v: 0 for v in singles}
    for u in singles:
        for v in out[u]:
            if v in indeg:
                indeg[v] += 1
    order, stack = [], [v for v in singles if indeg[v] == 0]
    while stack:
        u = stack.pop(); order.append(u)
        for v in out[u]:
            if v in indeg:
                indeg[v] -= 1
                if indeg[v] == 0:
                    stack.append(v)
    Ds = ''.join(sorted(D))
    return Ds + ''.join(letters[v] for v in order) + Ds

rng = random.Random(3)
for _ in range(300):
    al = 'abcdefg'[:rng.randint(1, 7)]
    w = rng.sample(sorted({a + b for a in al for b in al}), rng.randint(1, min(15, len(al) ** 2)))
    t = build_one_scs(w)
    assert all(is_subseq(x, t) for x in w) and len(t) == sum(Solution().supersequences(w)[0])
print('Bài 3: SCS dựng được hợp lệ; ví dụ 1 ->', build_one_scs(["ab", "ba"]))

Bài 1: [1, 1, 1] | Bài 2: [1, 2, 1]
Bài 3: SCS dựng được hợp lệ; ví dụ 1 -> aba


In [20]:
# Bài 4, 5
def acyclic_dfs(S, out_masks):
    color = {}
    def dfs(u):
        color[u] = 1
        r = out_masks[u] & S
        while r:
            v = (r & -r).bit_length() - 1; r &= r - 1
            c = color.get(v, 0)
            if c == 1 or (c == 0 and not dfs(v)):
                return False
        color[u] = 2
        return True
    r = S
    while r:
        v = (r & -r).bit_length() - 1; r &= r - 1
        if color.get(v, 0) == 0 and not dfs(v):
            return False
    return True

def count_min_D(words):
    letters, out, selfm = build(words)
    m = len(letters)
    inc = [0] * m
    for a in range(m):
        for b in out[a]:
            inc[b] |= 1 << a
    acyc = bytearray(1 << m); acyc[0] = 1
    for S in range(1, 1 << m):
        r = S
        while r:
            v = (r & -r).bit_length() - 1; r &= r - 1
            if inc[v] & S == 0 and acyc[S ^ (1 << v)]:
                acyc[S] = 1; break
    full = (1 << m) - 1
    sizes = [bin(D).count('1') for D in range(1 << m) if not (selfm & ~D) and acyc[full ^ D]]
    return sizes.count(min(sizes))

rng = random.Random(4)
for _ in range(200):
    al = 'abcdefgh'[:rng.randint(2, 8)]
    w = rng.sample(sorted({a + b for a in al for b in al}), rng.randint(1, min(20, len(al) ** 2)))
    letters, out, selfm = build(w)
    m = len(letters)
    outm = [sum(1 << v for v in out[u]) for u in range(m)]
    inc = [0] * m
    for a in range(m):
        for b in out[a]:
            inc[b] |= 1 << a
    for S in range(1 << m):
        assert acyclic_dfs(S, outm) == acyclic(S, inc)
    assert count_min_D(w) == len(Solution().supersequences(w))
print('Bài 4, 5 khớp; ví dụ 1 có', count_min_D(["ab", "ba"]), 'tập D nhỏ nhất.')

Bài 4, 5 khớp; ví dụ 1 có 2 tập D nhỏ nhất.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
inc[b] |= 1 << a  cho mỗi từ ab (a != b);  self |= 1 << a  cho từ aa
for size in 0..m:
    for D có |D| = size:
        if self ⊆ D and acyclic(full ^ D):   # Kahn trên bitmask
            thêm vectơ (2 cho D, 1 cho phần còn lại)
    if có kết quả: return
```

**Checklist:**

- [ ] Mỗi chữ 1 hoặc 2 lần (không cần thử 3)?
- [ ] Từ `xx` ⇒ `x` bắt buộc thuộc `D`?
- [ ] Chỉ lấy `D` có **cỡ nhỏ nhất**, nhưng lấy **mọi** `D` ở cỡ đó?
- [ ] Chữ không xuất hiện trong `words` có tần số 0?
- [ ] Kiểm tra chu trình trên **phần chữ đơn** `U \ D`?